# Mechanical Simulation with FiberNet

This tutorial covers mechanical simulation of fiber networks using FEM analysis:

1. Setting up a mechanical simulation
2. Running uniaxial tension tests
3. Extracting stress-strain curves
4. Computing mechanical properties
5. Comparing different network architectures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fibernet import gen
from fibernet.sim import FiberFEM
from fibernet.analysis import MorphologyAnalyzer, extract_stress_strain

## 1. Generate a Test Network

In [ ]:
# Create a random 2D network
net = gen.random_straight_2d(
    num_fibers=150,
    fiber_length=12.0,
    box_size=(50, 50),
    radius=0.1,
    seed=42
)

print(f'Network: {net.num_fibers} fibers, {net.num_crosslinks} crosslinks')

# Check network statistics
analyzer = MorphologyAnalyzer(net)
print(f'Mean length: {analyzer.mean_fiber_length():.2f}')
print(f'Nematic order: {analyzer.nematic_order_parameter():.3f}')
print(f'Porosity: {analyzer.porosity():.3f}')

## 2. Set Up FEM Solver

In [ ]:
# Create FEM solver
fem = FiberFEM(
    network=net,
    youngs_modulus=1e9,  # 1 GPa
    poisson_ratio=0.3,
    segments_per_fiber=10
)

print('FEM solver initialized')
print(f'Number of nodes: {fem.num_nodes}')
print(f'Number of elements: {fem.num_elements}')

## 3. Run Uniaxial Tension Test

In [ ]:
# Apply uniaxial tension in x-direction
result = fem.uniaxial_tension(
    strain=0.05,  # 5% strain
    axis=0,       # x-direction
    num_steps=20
)

print(f'Simulation completed: {len(result.strain)} strain steps')

## 4. Extract Stress-Strain Curve

In [ ]:
# Extract stress-strain data
stress_strain = extract_stress_strain(result)

# Plot stress-strain curve
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(stress_strain.strain, stress_strain.stress, 'b-', linewidth=2)
ax.set_xlabel('Strain', fontsize=12)
ax.set_ylabel('Stress (Pa)', fontsize=12)
ax.set_title('Uniaxial Tension Test', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Compute Mechanical Properties

In [ ]:
# Compute Young's modulus from initial slope
youngs_modulus = analyzer.compute_youngs_modulus(stress_strain)

# Compute yield strength
yield_strength = analyzer.compute_yield_strength(stress_strain, offset=0.002)

# Compute ultimate tensile strength
uts = analyzer.compute_ultimate_strength(stress_strain)

# Compute toughness (area under curve)
toughness = analyzer.compute_toughness(stress_strain)

print(f"Young's Modulus: {youngs_modulus/1e9:.2f} GPa")
print(f'Yield Strength: {yield_strength/1e6:.2f} MPa')
print(f'Ultimate Tensile Strength: {uts/1e6:.2f} MPa')
print(f'Toughness: {toughness/1e6:.2f} MJ/m³')

## 6. Compare Different Network Types

In [ ]:
# Generate different network types
networks = {
    'Random': gen.random_straight_2d(100, 10.0, (50, 50), seed=42),
    'Aligned': gen.oriented_2d(100, 10.0, (50, 50), orientation=0.0, seed=42),
    'Lattice': gen.square_lattice_2d(5.0, (10, 10)),
}

# Run simulations
fig, ax = plt.subplots(figsize=(10, 6))

for name, network in networks.items():
    fem = FiberFEM(network, youngs_modulus=1e9, poisson_ratio=0.3)
    result = fem.uniaxial_tension(strain=0.03, axis=0, num_steps=15)
    ss = extract_stress_strain(result)
    
    ax.plot(ss.strain, ss.stress, linewidth=2, label=name)

ax.set_xlabel('Strain', fontsize=12)
ax.set_ylabel('Stress (Pa)', fontsize=12)
ax.set_title('Comparison of Network Architectures', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Anisotropic Response

In [ ]:
# Test in different directions
directions = {'x-axis': 0, 'y-axis': 1}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (direction_name, axis) in enumerate(directions.items()):
    fem = FiberFEM(net, youngs_modulus=1e9, poisson_ratio=0.3)
    result = fem.uniaxial_tension(strain=0.04, axis=axis, num_steps=16)
    ss = extract_stress_strain(result)
    
    axes[i].plot(ss.strain, ss.stress, 'r-', linewidth=2)
    axes[i].set_xlabel('Strain', fontsize=12)
    axes[i].set_ylabel('Stress (Pa)', fontsize=12)
    axes[i].set_title(f'{direction_name} Loading', fontsize=13)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. GPU-Accelerated Simulation (Optional)

In [ ]:
# Use Taichi for GPU acceleration (requires taichi installed)
try:
    from fibernet.sim import TaichiFEMSolver
    
    # Create GPU-accelerated solver
    taichi_fem = TaichiFEMSolver(
        network=net,
        youngs_modulus=1e9,
        poisson_ratio=0.3
    )
    
    # Run simulation (much faster for large networks)
    result_gpu = taichi_fem.solve(
        strain=0.05,
        axis=0,
        num_steps=20
    )
    
    print('GPU simulation completed successfully')
    
except ImportError:
    print('Taichi not installed. Install with: pip install fibernet[accel]')

## Summary

In this tutorial, you learned how to:

- Set up FEM simulations for fiber networks
- Run uniaxial tension tests
- Extract and visualize stress-strain curves
- Compute mechanical properties (Young's modulus, yield strength, UTS, toughness)
- Compare different network architectures
- Analyze anisotropic mechanical response
- Use GPU acceleration for large networks

## Next Steps

- Explore `03_multi_physics.ipynb` for coupled simulations
- Try `04_machine_learning.ipynb` for property prediction
- Check the documentation for advanced constitutive models